# BazooAI — local Hugging Face model server on Colab

This notebook runs the DJ model server with **local inference only** using the public, ungated `Qwen/Qwen2.5-1.5B-Instruct` model. It never uses OpenRouter or a mock backend. If GPU/model loading/inference fails, a cell raises an error instead of silently switching models.

## Start
1. In Colab, select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU).
2. Add `NGROK_AUTH_TOKEN` to the Colab Secrets sidebar (needed to connect your local app). No Hugging Face or LLM API token is required for this public model.
3. Push the latest project changes to GitHub, set the repository URL/ref in the clone cell, and run the cells in order. The inference smoke test must pass before the server is considered ready.
4. Copy the printed URL into your local `.env` as `AI_MODEL_URL=...`, then restart Streamlit. Colab must remain running.


In [ ]:
import os
from google.colab import userdata

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN", "")
if not NGROK_AUTH_TOKEN:
    raise RuntimeError("Add NGROK_AUTH_TOKEN to Colab Secrets before continuing.")
os.environ["LLM_BACKEND"] = "local"
os.environ["HF_MODEL_NAME"] = MODEL_NAME
os.environ["HF_DTYPE"] = "float16"
os.environ["API_HOST"] = "0.0.0.0"
os.environ["API_PORT"] = "8000"
print(f"Local-only backend: {MODEL_NAME}")


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Select a GPU runtime and rerun.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
!pip install -q "transformers>=4.45,<5" accelerate sentencepiece protobuf fastapi "uvicorn[standard]" httpx "pydantic>=2" pyngrok pyyaml python-dotenv


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/10Unknownboy/BazooAI.git"
REPO_REF = "main"  # Set this to the branch containing your latest project changes.
PROJECT_DIR = Path("/content/BazooAI")
if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / "app/api/model_server.py").is_file():
    raise RuntimeError(f"Model server source is missing under {PROJECT_DIR}")
sys.path.insert(0, str(PROJECT_DIR))
print(f"Using project source: {PROJECT_DIR}")


In [ ]:
import os
from app.api.model_server import LLMBackend, app, parse_llm_json, router

if not hasattr(router, "local_only"):
    raise RuntimeError("This repository checkout predates local-only model mode. Push/update the project source and rerun.")
if os.environ.get("LLM_BACKEND") != "local":
    raise RuntimeError("The model server must run with LLM_BACKEND=local.")
print(f"Model server configured for local-only inference: {MODEL_NAME}")


In [ ]:
print("The local model is loaded once by FastAPI startup; the live inference smoke test follows server startup.")


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(addr=8000, proto="http")
public_url = tunnel.public_url
print(f"AI_MODEL_URL={public_url}")
print(f"Health check: {public_url}/health")


In [ ]:
import asyncio
import httpx
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
server_task = asyncio.create_task(server.serve())
async with httpx.AsyncClient(timeout=10) as client:
    for attempt in range(180):
        if server_task.done():
            await server_task
        try:
            response = await client.get("http://127.0.0.1:8000/health")
            response.raise_for_status()
            health = response.json()
            if health.get("active_backend") != "local":
                server.should_exit = True
                await server_task
                raise RuntimeError(f"Local model failed to load: {health}")
            print(f"Server ready with local model: {health['model_name']}")
            break
        except httpx.HTTPError:
            await asyncio.sleep(1)
    else:
        server.should_exit = True
        await server_task
        raise RuntimeError("FastAPI failed to become healthy with the local model")
raw, backend, error = await router.generate(
    'Return ONLY a JSON object with keys "ready" (boolean) and "model" (string). Set ready to true.',
    max_tokens=80,
)
parsed = parse_llm_json(raw)
if not backend.startswith("local:") or not parsed.get("ready"):
    server.should_exit = True
    await server_task
    raise RuntimeError(f"Local inference smoke test failed: backend={backend}, error={error}, output={raw[:300]!r}")
print(f"PASS: local inference via {backend}; parsed response: {parsed}")
print(f"Connect your local DJ app to {public_url}")


## Manual remote test

Run this after the server cell reports ready. It calls the public endpoint and fails unless the response came from a real local model.


In [ ]:
import httpx

payload = {
    "request_id": "COLAB_MANUAL_CHECK",
    "message_type": "DJ_DECISION",
    "candidate_songs": [
        {"song_id": "probe_1", "title": "Test Song", "artist": "Test Artist", "genre": "Pop", "energy": 0.7, "bpm": 120}
    ],
}
async with httpx.AsyncClient(timeout=90) as client:
    response = await client.post(f"{public_url}/v1/ai/decide", json=payload)
    response.raise_for_status()
    result = response.json()
if not result.get("success") or not str(result.get("model_name", "")).startswith("local:"):
    raise RuntimeError(f"Remote local-model check failed: {result}")
print(f"PASS: {result['model_name']} responded in {result.get('latency_ms', 0):.0f} ms")
print(f"Reason: {result.get('reason', '')}")


## Stop

To stop the server, run the following cell. The ngrok URL is valid only while the Colab runtime remains connected.


In [ ]:
server.should_exit = True
await server_task
ngrok.kill()
print("Server and tunnel stopped.")
